In [28]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv("../data/googleplaystore_clean.csv")

df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19.0,10000.0,Free,0.0,Everyone,Art & Design,"January 7, 2018"
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14.0,500000.0,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018"
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7,5000000.0,Free,0.0,Everyone,Art & Design,"August 1, 2018"
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25.0,50000000.0,Free,0.0,Teen,Art & Design,"June 8, 2018"
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8,100000.0,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018"


In [30]:
df["Last Updated"] = pd.to_datetime(df["Last Updated"])

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10356 entries, 0 to 10355
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   App             10356 non-null  str           
 1   Category        10356 non-null  str           
 2   Rating          10356 non-null  float64       
 3   Reviews         10356 non-null  float64       
 4   Size            10356 non-null  float64       
 5   Installs        10356 non-null  float64       
 6   Type            10356 non-null  str           
 7   Price           10356 non-null  float64       
 8   Content Rating  10356 non-null  str           
 9   Genres          10356 non-null  str           
 10  Last Updated    10356 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(5), str(5)
memory usage: 1.4 MB


In [32]:
latest_date = df["Last Updated"].max()

df["App_Age"] = (latest_date - df["Last Updated"]).dt.days
df[["Last Updated","App_Age"]].head()

,Last Updated,App_Age
0,2018-01-07,213
1,2018-01-15,205
2,2018-08-01,7
3,2018-06-08,61
4,2018-06-20,49


In [33]:
df["Review_Ratio"] = df["Reviews"] / df["Installs"]

In [34]:
df["Log_Reviews"] = np.log1p(df["Reviews"])

In [35]:
df["Log_Installs"] = np.log1p(df["Installs"])

In [36]:
df["Paid_App"] = df["Type"].map({
    "Free": 0,
    "Paid": 1
})

df["Paid_App"].value_counts()

Paid_App
0    9591
1     765
Name: count, dtype: int64

In [37]:
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,App_Age,Review_Ratio,Log_Reviews,Log_Installs,Paid_App
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19.0,10000.0,Free,0.0,Everyone,Art & Design,2018-01-07,213,0.015900,5.075174,9.210440,0
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14.0,500000.0,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,205,0.001934,6.875232,13.122365,0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7,5000000.0,Free,0.0,Everyone,Art & Design,2018-08-01,7,0.017502,11.379520,15.424949,0
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25.0,50000000.0,Free,0.0,Teen,Art & Design,2018-06-08,61,0.004313,12.281389,17.727534,0
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8,100000.0,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,49,0.009670,6.875232,11.512935,0


In [38]:
df.to_csv("../data/googleplaystore_features.csv", index=False)

In [39]:
import pandas as pd

apps = pd.read_csv("../data/googleplaystore_features.csv")
reviews = pd.read_csv("../data/googleplaystore_user_reviews.csv")

In [40]:
reviews.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [41]:
reviews = reviews.dropna(
    subset=["Sentiment_Polarity", "Sentiment_Subjectivity"]
)

In [42]:
review_summary = (
    reviews.groupby("App")
    .agg({
        "Sentiment_Polarity": "mean",
        "Sentiment_Subjectivity": "mean"
    })
    .reset_index()
)

In [43]:
review_summary.head()

,App,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,0.470733,0.495455
1,104 找工作 - 找工作 找打工 找兼職 履歷健檢 履歷診療室,0.392405,0.545516
2,11st,0.181294,0.443957
3,1800 Contacts - Lens Store,0.318145,0.591098
4,1LINE – One Line with One Touch,0.196290,0.557315


In [44]:
merged = apps.merge(
    review_summary,
    on="App",
    how="left"
)

In [45]:
merged["Sentiment_Polarity"] = merged["Sentiment_Polarity"].fillna(0)

merged["Sentiment_Subjectivity"] = merged["Sentiment_Subjectivity"].fillna(0)

In [46]:
merged.info()

merged.head()

<class 'pandas.DataFrame'>
RangeIndex: 10356 entries, 0 to 10355
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   App                     10356 non-null  str    
 1   Category                10356 non-null  str    
 2   Rating                  10356 non-null  float64
 3   Reviews                 10356 non-null  float64
 4   Size                    10356 non-null  float64
 5   Installs                10356 non-null  float64
 6   Type                    10356 non-null  str    
 7   Price                   10356 non-null  float64
 8   Content Rating          10356 non-null  str    
 9   Genres                  10356 non-null  str    
 10  Last Updated            10356 non-null  str    
 11  App_Age                 10356 non-null  int64  
 12  Review_Ratio            10342 non-null  float64
 13  Log_Reviews             10356 non-null  float64
 14  Log_Installs            10356 non-null  float64
 

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,App_Age,Review_Ratio,Log_Reviews,Log_Installs,Paid_App,Sentiment_Polarity,Sentiment_Subjectivity
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19.0,10000.0,Free,0.0,Everyone,Art & Design,2018-01-07,213,0.015900,5.075174,9.210440,0,0.000000,0.00000
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14.0,500000.0,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,205,0.001934,6.875232,13.122365,0,0.152652,0.64154
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7,5000000.0,Free,0.0,Everyone,Art & Design,2018-08-01,7,0.017502,11.379520,15.424949,0,0.000000,0.00000
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25.0,50000000.0,Free,0.0,Teen,Art & Design,2018-06-08,61,0.004313,12.281389,17.727534,0,0.000000,0.00000
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8,100000.0,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,49,0.009670,6.875232,11.512935,0,0.000000,0.00000


In [47]:
merged.to_csv("../data/merged_dataset.csv", index=False)